In [ ]:
!pip install czifile==2019.7.2 tifffile==2024.12.12 imagecodecs==2024.12.30

In [ ]:
import czifile as cz
import tifffile as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import statistics

def czi_to_tiff(input_path):
    # Create output filename
    output_path = os.path.splitext(input_path)[0] + ".tif"

    # Read CZI
    with cz.CziFile(input_path) as czi:
        image = czi.asarray()

    image = image.squeeze()  # remove extra dimensions

    # Save as TIFF
    tf.imwrite(output_path, image)

    print(f"Converted: {input_path} → {output_path}")

!wget https://char7927.github.io/BIOC0035/Data/galactoseA.czi
!wget https://char7927.github.io/BIOC0035/Data/galactoseB.czi
!wget https://char7927.github.io/BIOC0035/Data/galactoseC.czi
!wget https://char7927.github.io/BIOC0035/Data/glucoseA.czi
!wget https://char7927.github.io/BIOC0035/Data/glucoseB.czi
!wget https://char7927.github.io/BIOC0035/Data/glucoseC.czi


#List of your files
files = [
    "galactoseA.czi",
    "galactoseB.czi",
    "galactoseC.czi",
    "glucoseA.czi",
    "glucoseB.czi",
    "glucoseC.czi"
]

#Loop through and convert all
for file in files:
    czi_to_tiff(file)

In [ ]:
czi_files = ['glucoseA','glucoseB','glucoseC','galactoseA','galactoseB','galactoseC']
czi_ext = '.czi'
tif_ext = '.tiff'
num_imgs = 15
tiff_files = []

for i in range(len(czi_files)):
  #print(czi_files[i])
  #cz.czi2tif('galactoseA.czi','galactoseA.tiff')
  cz.czi2tif(czi_files[i]+czi_ext,czi_files[i]+tif_ext)
  tiff_files.append(czi_files[i]+tif_ext)

#img = tf.imread(czi_files[0]+tif_ext)
#tf.imshow(img[0])

  #cz.czi2tif('galactoseA.czi','galactoseA.tiff')
  #img = tf.imread('galactoseA.tiff')
  #tf.imshow(img[0])

In [ ]:
for tiff_file in tiff_files:
  print(tiff_file)
  img_file = tf.imread(tiff_file)
  tmp = 30
  for i in range(num_imgs):
    print('timepoint =',tmp, 'seconds')
    plt.imshow(img_file[i])
    plt.show()
    tmp = tmp + 30

In [ ]:
col_img = tf.imread(tiff_files[0])

tf.imshow(col_img[0])

In [ ]:
np_img = tf.imread(tiff_files[0])
np_img0_flat = np.ndarray.flatten(np_img[0])

plt.figure(figsize=(16,5))
n_img, bins_img, _ = plt.hist(np_img0_flat,np.max(np_img0_flat)+1,(-0.5,np.max(np_img0_flat)+0.5))
plt.axvline(175,color='red')
plt.axvline(350,color='red')
plt.xlabel('Pixel brightness')
print(n_img[175:350])
print(np.argmin(n_img[175:350]))
print(n_img[256])
plt.axvline(256,color='green')

In [ ]:
# Create a figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Set a main overarching title for the whole figure
fig.suptitle('Baseline Galactose A (30s)')

# Plot the raw image in the first slot
axes[0].imshow(np_img[0])
axes[0].set_title('Raw Confocal Image')

# Generate and plot the mask in the second slot
mask_np_img0 = np.where(np_img[0] > 256, 1, 0)
axes[1].imshow(mask_np_img0, cmap='gray')
axes[1].set_title('Binary Segmentation Mask')

# Display them together
plt.show()

In [ ]:
# 1. Load the Glucose A file (the 4th file in your list)
np_img_glucose = tf.imread(tiff_files[3])

# 2. Create a figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Baseline Glucose A (30s)')

# 3. Plot the raw image for the first timepoint [0]
axes[0].imshow(np_img_glucose[0])
axes[0].set_title('Raw Confocal Image')

# 4. Generate and plot the mask for that same timepoint
mask_glucose = np.where(np_img_glucose[0] > 256, 1, 0)
axes[1].imshow(mask_glucose, cmap='gray')
axes[1].set_title('Binary Segmentation Mask')

plt.show()

In [ ]:
plt.imshow(np_img[0])
plt.show()
mask_np_img0 = np.where(np_img[0] > 256,1,0)
plt.imshow(mask_np_img0,cmap='gray')

In [ ]:
threshholds = [700,600,800,650,720,750]
def buildMask(indx):
  test_img_file = tf.imread(tiff_files[indx])
  comb_mask = np.where(test_img_file[0] > threshholds[indx], 1,0)
  #print('using threshhold:',threshholds[indx])
  #print(np.sum(comb_mask))
  for i in range(1,5):
    add_mask = np.where(test_img_file[1] > 700, 1,0)
    #print(np.sum(add_mask))
    comb_mask = comb_mask + add_mask
    #print('new comb_mask:' ,np.sum(comb_mask))
  #print(comb_mask[58][35:55])
  comb_mask = np.where(comb_mask > 0,1,0)
  #print(comb_mask[58][35:55])
  plt.imshow(comb_mask, cmap = 'gray')
  plt.show()
  return comb_mask

plt.imshow((tf.imread(tiff_files[1]))[0])
plt.show()
msk = buildMask(1)

print(msk.shape)

In [ ]:
#calc avg brightness for each image
avgs = []

for i in range(len(tiff_files)):
  tiff_file = tf.imread(tiff_files[i])
  msk = buildMask(i)
  these_avgs = []

  for j in range(num_imgs):
    img = tiff_file[j]
    filt_img = img[msk > 0]
    mn = np.mean(filt_img)
    these_avgs.append(mn)

  avgs.append(these_avgs)

print(len(avgs))
print(len(avgs[0]))

In [ ]:
#use for graphs
timepoints = [30,60,90,120,150,180,210,240,270,300,330,360,390,420,450]

for i in range(len(tiff_files)):
  plt.plot(timepoints, avgs[i])
  plt.title('Flourescence intensities of sugars')
  plt.xlabel('Time (s)')
  plt.ylabel('Brightness (au)')
  plt.legend(czi_files)


In [ ]:
percs = []
timepoints = [30,60,90,120,150,180,210,240,270,300,330,360,390,420,450]

for i in range(len(tiff_files)):
  fccp = np.mean(avgs[i][6:10])
  rot = np.mean(avgs[i][11:15])
  these_percs = []

  for j in range(num_imgs):
    perc = (avgs[i][j]-fccp)/(rot-fccp)*100
    these_percs.append(perc)

  percs.append(these_percs)

print(percs)

for i in range(len(tiff_files)):
  plt.plot(timepoints, percs[i])
  plt.title('Percentage NADH of sugars')
  plt.xlabel('Time (s)')
  plt.ylabel('Percentage of NADH')
  plt.legend(czi_files)

In [ ]:
avg_nadhs = ([],[])
stdev_nadhs = ([],[])

for i in range(num_imgs):
  print(percs[0][i],percs[1][i],percs[2],[i])
  avg_nadhs[0].append(np.mean([percs[0][i],percs[1][i],percs[2][i]]))
  avg_nadhs[1].append(np.mean([percs[3][i],percs[4][i],percs[5][i]]))
  stdev_nadhs[0].append(statistics.stdev([percs[0][i],percs[1][i],percs[2][i]]))
  stdev_nadhs[1].append(statistics.stdev([percs[3][i],percs[4][i],percs[5][i]]))

print(avg_nadhs)
print(stdev_nadhs)

plt.errorbar(timepoints, avg_nadhs[0], yerr=stdev_nadhs[0],ecolor= 'brown')
plt.errorbar(timepoints, avg_nadhs[1], yerr=stdev_nadhs[1],ecolor= 'brown')
plt.title('Average NADH percentage of sugar samples')
plt.ylabel('Mean NADH percentage')
plt.xlabel('Time (s)')
plt.legend(['Glucose', 'Galactose'],loc='upper left')

plt.hlines(-10,175,300, color= 'black')
plt.text(226, -8, 'FCCP')

print('Glucose:',np.mean(avg_nadhs[0][:5]))
print('Galactose:',np.mean(avg_nadhs[1][:5]))

In [ ]:
# 1. Calculate the overall mean of the 5 baseline intervals (0 to 150 seconds)
glu_baseline_mean = np.mean(avg_nadhs[0][:5])
gal_baseline_mean = np.mean(avg_nadhs[1][:5])

# 2. Calculate the standard deviation of those 5 baseline intervals for the error bars
glu_baseline_std = np.std(avg_nadhs[0][:5])
gal_baseline_std = np.std(avg_nadhs[1][:5])

# 3. Set up the data for plotting
conditions = ['Glucose (25mM)', 'Galactose (10mM)']
means = [glu_baseline_mean, gal_baseline_mean]
stdevs = [glu_baseline_std, gal_baseline_std]

# 4. Create the bar chart
plt.figure(figsize=(6, 5))
# Using standard matplotlib blue and orange to match your line graph, with black edges for a professional look
plt.bar(conditions, means, yerr=stdevs, capsize=5, color=['#1f77b4', '#ff7f0e'], edgecolor='black')

# 5. Add formatting (Title is commented out for the lab report export!)
# plt.title('Average Baseline NADH Percentage')
plt.ylabel('Mean NADH Percentage')
plt.ylim(0, 70) # Gives a little breathing room above the 51.5% bar

# Display the clean graph ready for export
plt.show()